Creation of Catalog and Schema's

In [0]:
CATALOG = "payghana"  

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.raw")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
print(f"Catalog and schemas ready under {CATALOG}")

Generate synthetic KYC data and load Raw


In [0]:
import random
import pandas as pd
%pip install faker
from faker import Faker


fake = Faker()
GHANA_FIRST = ["Kwame", "Kofi", "Ama", "Akosua", "Yaw", "Efua", "Kojo", "Nana", "Adwoa"]
GHANA_LAST = ["Mensah", "Owusu", "Boateng", "Osei", "Asare", "Addo", "Appiah"]

def gen_gh_card() -> str:
    return f"GHA-{random.randint(100000000, 999999999)}-{random.randint(0, 9)}"

def gen_phone() -> str:
    return f"0{random.choice(['24', '54', '55', '20'])}{random.randint(1000000, 9999999)}"

customers = [
    {
        "customer_id": f"CUST{i}",
        "first_name": random.choice(GHANA_FIRST),
        "last_name": random.choice(GHANA_LAST),
        "ghana_card_no": gen_gh_card(),
        "phone": gen_phone(),
        "region": random.choice(["Greater Accra", "Ashanti"]),
    }
    for i in range(1000)
]

df_bronze_pd = pd.DataFrame(customers)
sdf_bronze = spark.createDataFrame(df_bronze_pd)
sdf_bronze.write.mode("overwrite").saveAsTable(f"{CATALOG}.raw.kyc_customers")
print(f"Raw/Bronze loaded: {CATALOG}.raw.kyc_customers ({sdf_bronze.count()} rows)")

In [0]:
%restart_python

Validate and load Silver

In [0]:
from pyspark.sql import functions as F

sdf_silver = (
    sdf_bronze.dropDuplicates(["ghana_card_no"])
    .dropna(subset=["ghana_card_no", "phone", "customer_id"])
    .withColumn("phone", F.regexp_replace("phone", " ", ""))
    .filter(F.col("phone").rlike(r"^0(24|54|55|20)\d{7}$"))
    .filter(F.col("ghana_card_no").rlike(r"^GHA-\d{9}-\d$"))
    .withColumn("region", F.initcap(F.trim(F.col("region"))))
)

sdf_silver.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.kyc_customers")
print(f"Silver loaded — {sdf_silver.count()} validated rows")

Create the Gold view


In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {CATALOG}.gold.kyc_customers_secure AS
    SELECT * FROM {CATALOG}.silver.kyc_customers
""")
print("Gold view created")

Create the column mask and row filter


In [0]:
%sql
CREATE OR REPLACE FUNCTION payghana.gold.mask_ghana_card(val STRING)
RETURN
  IF(
    IS_ACCOUNT_GROUP_MEMBER('compliance_leads'),
    val,
    CONCAT(LEFT(val, 4), '*******', RIGHT(val, 1))
  );

ALTER TABLE payghana.silver.kyc_customers
  ALTER COLUMN ghana_card_no SET MASK payghana.gold.mask_ghana_card;

CREATE OR REPLACE FUNCTION payghana.gold.region_filter(region STRING)
RETURN
  IS_ACCOUNT_GROUP_MEMBER('risk_analysts')
  OR IS_ACCOUNT_GROUP_MEMBER('compliance_leads')
  OR (IS_ACCOUNT_GROUP_MEMBER('compliance_team') AND region = 'Greater Accra');

ALTER TABLE payghana.silver.kyc_customers
  SET ROW FILTER payghana.gold.region_filter ON (region);

Create the column mask and row filter

In [0]:
%sql
CREATE OR REPLACE FUNCTION payghana.gold.mask_ghana_card(val STRING)
RETURN
  IF(
    IS_ACCOUNT_GROUP_MEMBER('compliance_leads'),
    val,
    CONCAT(LEFT(val, 4), '*******', RIGHT(val, 1))
  );

ALTER TABLE payghana.silver.kyc_customers
  ALTER COLUMN ghana_card_no SET MASK payghana.gold.mask_ghana_card;

CREATE OR REPLACE FUNCTION payghana.gold.region_filter(region STRING)
RETURN
  IS_ACCOUNT_GROUP_MEMBER('risk_analysts')
  OR IS_ACCOUNT_GROUP_MEMBER('compliance_leads')
  OR (IS_ACCOUNT_GROUP_MEMBER('compliance_team') AND region = 'Greater Accra');

ALTER TABLE payghana.silver.kyc_customers
  SET ROW FILTER payghana.gold.region_filter ON (region);

Create the role-scoped views and grant access

In [0]:
%sql
CREATE OR REPLACE VIEW payghana.gold.v_risk_analyst AS
SELECT customer_id, first_name, last_name, phone, region
FROM payghana.silver.kyc_customers;

GRANT SELECT ON VIEW payghana.gold.v_risk_analyst TO `risk_analysts`;

CREATE OR REPLACE VIEW payghana.gold.v_compliance_accra AS
SELECT customer_id, first_name, last_name, ghana_card_no, phone, region
FROM payghana.silver.kyc_customers;

GRANT SELECT ON VIEW payghana.gold.v_compliance_accra TO `compliance_team`;

Run classification and test the access split

In [0]:
%sql

SELECT * FROM payghana.gold.v_compliance_accra;

In [0]:
%sql
SELECT
  is_member('risk_analysts')    AS in_risk_analysts,
  is_member('compliance_team')  AS in_compliance_team,
  is_member('compliance_leads') AS in_compliance_leads;

In [0]:
%sql
DESCRIBE TABLE EXTENDED payghana.silver.kyc_customers;

In [0]:
%sql
SELECT
  payghana.gold.region_filter('Ashanti')       AS ashanti_result,
  payghana.gold.region_filter('Greater Accra') AS accra_result;

In [0]:
%sql
SELECT * FROM payghana.gold.v_compliance_accra;

In [0]:
%sql
SELECT
  is_member('risk_analysts')    AS in_risk_analysts,
  is_member('compliance_team')  AS in_compliance_team,
  is_member('compliance_leads') AS in_compliance_leads;

In [0]:
%sql
SELECT * FROM payghana.gold.v_risk_analyst;

In [0]:
%sql
SELECT * FROM payghana.gold.v_compliance_accra;

Enable system tables for audit

In [0]:
%sql
SELECT event_time, user_identity.email AS user_email, request_params.full_name_arg AS table_name, action_name
FROM system.access.audit
WHERE request_params.full_name_arg LIKE '%kyc_customers%'
  AND event_time >= current_timestamp() - INTERVAL 30 DAYS
ORDER BY event_time DESC;